In [2]:
include("../RayTracing.jl")

Main.RayTracing

In [27]:
env_light = RayTracing.InfinteLight(
    RayTracing.Bounds3(RayTracing.Pnt3(-5, -5, -5), RayTracing.Pnt3(5, 5, 5)), 
    RayTracing.Translate(RayTracing.Vec3(0,0,0)), 
    RayTracing.Spectrum(1.0), 
    "../../ref/sky.exr"
)

Main.RayTracing.InfinteLight(Main.RayTracing.LightInfinite, Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 -0.0; 0.0 1.0 0.0 -0.0; 0.0 0.0 1.0 -0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 -0.0; 0.0 1.0 0.0 -0.0; 0.0 0.0 1.0 -0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), [1.0, 1.0, 1.0], Main.RayTracing.Distribution2D(Main.RayTracing.Distribution1D[Main.RayTracing.Distribution1D([5.542702795991198e-5, 9.237834370397201e-5, 0.00012932960510423253, 0.00016628079042321846, 0.00020323187792349953, 0.000240182845867703, 0.0002771336725185264, 0.0003140843361387504, 0.0003510348149912517, 0.0003879850873390156  …  0.00031408433613875725, 0.0002771336725185293, 0.00024018284586770203, 0.00020323187792351612, 0.00016628079042323117, 0.00012932960510424134, 9.237834370397693e-5, 5.5427027959913e-5, 1.8475679609556567e-5, -1.847567960954477e-5], [0.0, 4.410350

In [28]:
X, Y = size(env_light.map)
m = 0.0
for x in 1:X
    for y in 1:Y
        MAX = max(env_light.map[x,y].r, env_light.map[x,y].g, env_light.map[x,y].b)
        if MAX > m
            m = MAX
        end
    end
end
print(m)

2.032e4

In [34]:
typeof(env_light.map)

Matrix{RGBA{Float16}} (alias for Array{ColorTypes.RGBA{Float16}, 2})

In [45]:
x, y = size(env_light.map)
maximum(Float64.(reinterpret(Float16, RayTracing.Gray.(env_light.map))))

19856.0

In [49]:
Float64.(RayTracing.Gray.(env_light.map))

19856.0

In [89]:
x, y = size(env_light.map)
u, uv = findmax(Float64.(RayTracing.Gray.(env_light.map)))

(19856.0, CartesianIndex(569, 1024))

In [71]:
uv[1]/x, uv[2]/y

(0.27783203125, 0.25)

In [92]:
(u, v), pdf_val = RayTracing.sample_continuous(env_light.pdf, RayTracing.Pnt2(0.5, 0.5))

([0.2500023667131565, 0.2780281699157606], 171917.03873114358)

In [95]:
function _fuck(a::Float64, b::Int64)::Int64
    return max(1,Int(trunc(a * b)))
end

_fuck (generic function with 1 method)

In [96]:
sample = env_light.map[_fuck(v, x), _fuck(u, y)]
sample.r, sample.g, sample.b

(Float16(1.9e4), Float16(2.032e4), Float16(1.968e4))

In [8]:
r = RayTracing.Pnt2(.5, .5)

2-element Main.RayTracing.Pnt2 with indices SOneTo(2):
 0.5
 0.5

# sample_li

In [9]:
uv, map_pdf = RayTracing.sample_continuous(env_light.pdf, r)
print("uv: ", uv, "\n")
print("map_pdf: ", map_pdf, "\n")

theta = uv.y * pi
phi = uv.x * 2 * pi
print("phi, theta: ", phi, ", ", theta, "\n")

cos_theta = RayTracing.cos(theta)
sin_theta = RayTracing.sin(theta)
sin_phi = RayTracing.sin(phi)
cos_phi = RayTracing.cos(phi)
wi = env_light.light_to_world(RayTracing.Vec3(sin_theta * cos_phi, sin_theta * sin_phi, cos_theta))
print("wi: ", wi, "\n")

map_pdf /= (2 * pi * pi * sin_theta)
print("map_pdf: ", map_pdf, "\n")

x, y = size(env_light.map)
uu = RayTracing.Int(RayTracing.trunc(uv.x * x)+1)
vv = RayTracing.Int(RayTracing.trunc(uv.x * x)+1)
radiance = env_light.map[uu, vv]
print("radiance: ", radiance.r, ", ", radiance.g, ", ", radiance.b, "\n")

uv: [0.2500023667131565, 0.2780281699157606]
map_pdf: 171917.03873114358


phi, theta: 1.5708111972922278, 0.8734512560983683
wi: 

[-1.1398977381377058e-5, 0.7665498420554099, 0.6421848172565974]
map_pdf: 11361.842958503961
radiance: 0.04074, 0.09467, 0.1324


# pdf_li

In [11]:
wi = env_light.world_to_light(wi)
theta = RayTracing.spherical_theta(wi)
phi = RayTracing.spherical_phi(wi)
sin_theta = RayTracing.sin(theta)
(sin_theta == 0.0) && return 0.0

u_idx = phi / 2pi
v_idx = theta / pi
print("uv: ", u_idx, ", ", v_idx, "\n")

pdf_val = RayTracing.pdf(env_light.pdf, RayTracing.Pnt2(u_idx, v_idx))
print("pdf_map: ", pdf_val, "\n")
print("pdf_map: ", pdf_val / (2 * pi * pi * sin_theta), "\n")

uv: 0.2500023667131565, 0.2780281699157606
pdf_map: 171917.03873114358
pdf_map: 11361.842958503961
